# Estimación FOPDT 
Este notebook realiza una versión mínima: carga datos, estima un modelo FOPDT con tres métodos (Ziegler–Nichols, Smith y optimización) y muestra gráficos y parámetros (K, τ, θ).


## Entrada de datos
El notebook intenta detectar automáticamente columnas de tiempo, entrada y salida. Cambie `data_file`, `u_col` o `y_col` si desea otra columna del fichero.

In [ ]:
# Imports básicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# Ruta por defecto (ajustable)
data_file = 'lab 1/rta/data_Q1_Q2.txt'
data_file = 'data_Q1.txt'
# Columnas por defecto: se elige automáticamente si encuentra nombres evidentes
u_col = None
y_col = None
t_col = None

# Cargar con pandas (comas como separador y encabezado en la primera línea)
# Usar encoding latin1 para manejar símbolos como ° o codificaciones locales
df = pd.read_csv(data_file, sep=',', header=0, comment='#', encoding='latin1')
cols = list(df.columns)
print('Columnas detectadas:', cols)
# Selección automática heurística
for c in cols:
    cl = c.lower()
    if 'tiempo' in cl or 'time' in cl:
        t_col = c
    if u_col is None and ('cal' in cl or 'set' in cl or 'heater' in cl or 'input' in cl):
        u_col = c
    if y_col is None and ('temp' in cl or 'temper' in cl or 'y' == cl.strip()):
        y_col = c
# Fallbacks
if t_col is None:
    t_col = cols[0]
if u_col is None:
    # intentar segunda columna
    u_col = cols[1] if len(cols) > 1 else cols[0]
if y_col is None:
    # intentar tercera o cuarta columna
    y_col = cols[3] if len(cols) > 3 else cols[-1]

# Extraer arrays
t = df[t_col].values.astype(float)
u = df[u_col].values.astype(float)
y = df[y_col].values.astype(float)

# Detección de escalón en la entrada (primer cambio significativo)
du = np.diff(u)
thres = (np.nanmax(u) - np.nanmin(u)) * 0.1
step_idx_candidates = np.where(np.abs(du) > thres)[0]
if len(step_idx_candidates) == 0:
    step_idx = 0
else:
    step_idx = step_idx_candidates[0] + 1
t_step = t[step_idx]
print(f'Usando columnas: tiempo=`{t_col}`, entrada=`{u_col}`, salida=`{y_col}`')
print('Índice de escalón detectado:', step_idx, 'tiempo escalón=', t_step)

# Valores inicial y final (promedios)
y0 = np.mean(y[max(0, step_idx-20):step_idx])
y_ss = np.mean(y[-20:])
u0 = np.mean(u[max(0, step_idx-20):step_idx])
u_ss = np.mean(u[-20:])
delta_u = u_ss - u0 if (u_ss - u0) != 0 else 1.0
print('delta_u =', delta_u)

# Gráfica rápida de señal de entrada y salida
plt.figure(figsize=(8,3))
plt.plot(t, y, label='Salida')
plt.plot(t, u, label='Entrada (u)')
plt.axvline(t_step, color='k', linestyle='--', alpha=0.5)
plt.legend()
plt.xlabel('Tiempo (s)')
plt.tight_layout()
plt.show()

# Guardar variables en el entorno para las siguientes celdas
data = dict(t=t, u=u, y=y, t_step=t_step, y0=y0, y_ss=y_ss, u0=u0, u_ss=u_ss, delta_u=delta_u)

## 6.1 Ziegler–Nichols (método de la tangente)
Procedimiento:
1) Trazar tangente en el punto de máxima pendiente de la respuesta.
2) Calcular intersección de esa tangente con la línea del valor inicial y con el valor final para obtener θ y una aproximación de τ.
3) Calcular K = Δy / Δu.

In [ ]:
# Implementación del método de la tangente (Ziegler–Nichols)
t = data['t']
y = data['y']
u = data['u']
t_step = data['t_step']
y0 = data['y0']
y_ss = data['y_ss']
delta_u = data['delta_u']

# derivada numérica
dy_dt = np.gradient(y, t)
# suavizar derivada (media móvil corta)
window = max(1, int(len(dy_dt)*0.01))
if window > 1:
    dy_dt_s = np.convolve(dy_dt, np.ones(window)/window, mode='same')
else:
    dy_dt_s = dy_dt

i_inf = np.nanargmax(np.abs(dy_dt_s))
t_inf = t[i_inf]
y_inf = y[i_inf]
slope = dy_dt_s[i_inf]

# Tangente: y = y_inf + slope*(t - t_inf)
# Intersección con valor inicial y final
t_intersect_init = t_inf + (y0 - y_inf)/slope
t_intersect_final = t_inf + (y_ss - y_inf)/slope
theta_zn = max(0.0, t_intersect_init - t_step)
tau_zn = max(1e-6, t_intersect_final - t_intersect_init)
K_zn = (y_ss - y0) / delta_u

print('Ziegler–Nichols estimado: K={:.4f}, tau={:.4f}, theta={:.4f}'.format(K_zn, tau_zn, theta_zn))

# Gráfica con tangente y puntos de intersección
plt.figure(figsize=(8,3))
plt.plot(t, y, label='Salida medida')
# tangente
y_tan = y_inf + slope*(t - t_inf)
plt.plot(t, y_tan, '--', label='Tangente (inflection)')
plt.axvline(t_step + theta_zn, color='C2', linestyle=':', label='theta (ZN)')
plt.scatter([t_inf, t_intersect_init, t_intersect_final], [y_inf, y0, y_ss], c=['C3','C4','C5'])
plt.legend()
plt.xlabel('Tiempo (s)')
plt.title('Ziegler–Nichols: tangente en inflexión')
plt.tight_layout()
plt.show()

# Guardar estimado
zn = dict(K=K_zn, tau=tau_zn, theta=theta_zn)
data['zn'] = zn

## 6.2 Smith
Procedimiento:
- Se traza la misma tangente y se obtiene θ como intersección.
- τ se estima como el tiempo que tarda en alcanzar el 63.2% del cambio final después de la demora (método 0.632).

**Explicación (por qué 63.2%)**:
Para un sistema FOPDT la respuesta al escalón es $y(t)=y_0+K\Delta u(1-e^{-(t-\theta)/\tau})$ para $t>\theta$. Evaluando en $t=\theta+\tau$ se obtiene $1-e^{-1}\approx0.632$. Por eso se usa el 63.2%: el tiempo en que la salida alcanza ese porcentaje (restando la demora) ofrece una estimación directa de $\tau$.

Nota: en datos ruidosos o con muestreo escaso conviene suavizar la señal o elegir la intersección más representativa; en este notebook usamos $\theta$ de la tangente y luego el cruce al 63.2% para calcular $\tau$.

In [ ]:
# Método Smith (tangente + 63.2%)
# Reusar theta de la tangente (puede ajustarse manualmente)
theta_smith = data['zn']['theta']
y_target = y0 + 0.632*(y_ss - y0)
# buscar tiempo en que la señal cruza el 63.2% (primera vez)
idx_632 = np.where(y >= y_target)[0]
if len(idx_632) == 0:
    t_632 = t_inf + data['zn']['tau']
else:
    t_632 = t[idx_632[0]]
tau_smith = max(1e-6, t_632 - (t_step + theta_smith))
K_smith = (y_ss - y0) / delta_u

print('Smith estimado: K={:.4f}, tau={:.4f}, theta={:.4f}'.format(K_smith, tau_smith, theta_smith))

# Gráfica
plt.figure(figsize=(8,3))
plt.plot(t, y, label='Salida medida')
plt.axvline(t_step + theta_smith, color='C2', linestyle=':', label='theta (Smith)')
plt.axhline(y_target, color='C3', linestyle='--', label='63.2%')
plt.legend()
plt.xlabel('Tiempo (s)')
plt.title('Smith: 0.632 method')
plt.tight_layout()
plt.show()

smith = dict(K=K_smith, tau=tau_smith, theta=theta_smith)
data['smith'] = smith

## 6.3 Comparación y Optimización (identificación por mínimos cuadrados)
Usamos los estimados anteriores como iniciales para una optimización no lineal que minimiza el error entre la respuesta del FOPDT y los datos experimentales.

In [ ]:
# Modelo FOPDT vectorizado para respuesta a escalón
def fopdt_response(t, K, tau, theta, t_step, u0, delta_u, y0):
    t_rel = t - t_step - theta
    y_mod = np.where(t_rel>0, y0 + K*delta_u*(1.0 - np.exp(-t_rel / tau)), y0)
    return y_mod

# Residuals para least_squares
def residuals(p, t, y, t_step, u0, delta_u, y0):
    K, tau, theta = p
    if tau <= 0 or theta < 0:
        return 1e6 * np.ones_like(y)
    y_mod = fopdt_response(t, K, tau, theta, t_step, u0, delta_u, y0)
    return y_mod - y

# Condiciones iniciales a partir de Smith por defecto
p0 = [data['smith']['K'], data['smith']['tau'], data['smith']['theta']]
# límites razonables: K libre, tau>1e-3, theta>=0
lb = [-np.inf, 1e-6, 0.0]
ub = [np.inf, t[-1]*2, t[-1]]

res = least_squares(residuals, p0, bounds=(lb, ub), args=(t, y, t_step, data['u0'], data['delta_u'], data['y0']))
K_opt, tau_opt, theta_opt = res.x
print('Optimización (least_squares): K={:.4f}, tau={:.4f}, theta={:.4f}'.format(K_opt, tau_opt, theta_opt))

# Gráfica comparativa
y_fit = fopdt_response(t, K_opt, tau_opt, theta_opt, t_step, data['u0'], data['delta_u'], data['y0'])
plt.figure(figsize=(8,3))
plt.plot(t, y, label='Salida medida')
plt.plot(t, y_fit, '--', label='FOPDT optimizado')
plt.legend()
plt.xlabel('Tiempo (s)')
plt.title('Comparación: datos vs FOPDT optimizado')
plt.tight_layout()
plt.show()

data['opt'] = dict(K=K_opt, tau=tau_opt, theta=theta_opt)

## Resumen de estimaciones
Se imprimen las tres estimaciones para fácil comparación.

In [ ]:
print('Ziegler–Nichols: ', data['zn'])
print('Smith: ', data['smith'])
print('Optimización: ', data['opt'])

In [ ]:
print("=== Ziegler–Nichols ===")
print("K     :", data['zn']['K'])
print("tau   :", data['zn']['tau'])
print("theta :", data['zn']['theta'])

print("\n=== Smith ===")
print("K     :", data['smith']['K'])
print("tau   :", data['smith']['tau'])
print("theta :", data['smith']['theta'])

print("\n=== Optimización ===")
print("K     :", data['opt']['K'])
print("tau   :", data['opt']['tau'])
print("theta :", data['opt']['theta'])